# Task 2: Channel-Level Network Analysis

**Objective:** Build a channel-to-channel network based on tag overlap and temporal co-posting patterns, detect communities, and test whether coordinated clusters exhibit significantly higher engagement (H1).

**Data source:** BigQuery (`infinite-rope-363317.youtube_data`)

## 1. Setup & Data Loading

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import networkx as nx
import community as community_louvain
import plotly.graph_objects as go
import plotly.express as px
from collections import defaultdict
from itertools import combinations
from scipy import stats
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from utils.bq_helpers import load_videos, load_channels, COLORS

print('Libraries loaded successfully')

Libraries loaded successfully


In [2]:
# Load data
channels = load_channels()
print(f'Channels: {channels.shape}')

videos = load_videos(['channel_id', 'video_id', 'published_at', 'tags', 'view_count', 'like_count', 'comment_count'])
print(f'Videos: {videos.shape}')

print(f'\nUnique channels in videos: {videos["channel_id"].nunique()}')
print(f'Videos with tags: {videos["tags"].notna().sum():,} ({videos["tags"].notna().mean()*100:.1f}%)')

Channels: (1561, 11)


Videos: (88585, 7)

Unique channels in videos: 263
Videos with tags: 52,004 (58.7%)


## 2. Network Construction

Two edge types:
1. **Tag overlap**: Jaccard similarity of channel tag sets (threshold > 0.15)
2. **Temporal co-posting**: Days where both channels posted (threshold >= 5 days)

In [3]:
# Build per-channel tag sets
def parse_tags(tag_str):
    if pd.isna(tag_str):
        return set()
    return set(t.strip().lower() for t in str(tag_str).split('|') if t.strip())

channel_tags = {}
for cid, group in videos.groupby('channel_id'):
    all_tags = set()
    for t in group['tags'].dropna():
        all_tags.update(parse_tags(t))
    if all_tags:
        channel_tags[cid] = all_tags

print(f'Channels with tags: {len(channel_tags)}')

# Build inverted index: tag -> set of channels
tag_to_channels = defaultdict(set)
for cid, tags in channel_tags.items():
    for tag in tags:
        tag_to_channels[tag].add(cid)

print(f'Unique tags: {len(tag_to_channels):,}')

Channels with tags: 245


Unique tags: 146,173


In [4]:
# Compute Jaccard similarity only for channel pairs sharing at least 1 tag
candidate_pairs = set()
for tag, ch_set in tag_to_channels.items():
    if len(ch_set) > 1 and len(ch_set) < 200:  # skip overly common tags
        for pair in combinations(ch_set, 2):
            candidate_pairs.add(tuple(sorted(pair)))

print(f'Candidate pairs to evaluate: {len(candidate_pairs):,}')

# Compute Jaccard for candidates
tag_edges = {}
for c1, c2 in tqdm(candidate_pairs, desc='Computing Jaccard'):
    t1, t2 = channel_tags[c1], channel_tags[c2]
    jaccard = len(t1 & t2) / len(t1 | t2)
    if jaccard > 0.15:
        tag_edges[(c1, c2)] = jaccard

print(f'Tag-overlap edges (Jaccard > 0.15): {len(tag_edges):,}')

Candidate pairs to evaluate: 9,733


Computing Jaccard:   0%|          | 0/9733 [00:00<?, ?it/s]

Computing Jaccard:  24%|██▍       | 2355/9733 [00:00<00:00, 23538.69it/s]

Computing Jaccard:  50%|████▉     | 4840/9733 [00:00<00:00, 24301.49it/s]

Computing Jaccard:  76%|███████▌  | 7392/9733 [00:00<00:00, 24852.51it/s]

Computing Jaccard: 100%|██████████| 9733/9733 [00:00<00:00, 24649.67it/s]

Tag-overlap edges (Jaccard > 0.15): 0


In [5]:
# Temporal co-posting edges
videos['post_date'] = videos['published_at'].dt.date
channel_dates = videos.groupby('channel_id')['post_date'].apply(set).to_dict()

# Only check pairs that have overlapping date ranges
channel_ids_list = list(channel_dates.keys())
temporal_edges = {}

# Focus on channels with tags for efficiency
if len(channel_ids_list) > 500:
    active_channels = [c for c in channel_ids_list if c in channel_tags]
else:
    active_channels = channel_ids_list

print(f'Computing temporal co-posting for {len(active_channels)} channels...')

for i in tqdm(range(len(active_channels)), desc='Temporal edges'):
    for j in range(i+1, len(active_channels)):
        c1, c2 = active_channels[i], active_channels[j]
        shared = len(channel_dates.get(c1, set()) & channel_dates.get(c2, set()))
        if shared >= 5:
            pair = tuple(sorted((c1, c2)))
            temporal_edges[pair] = shared

print(f'Temporal co-posting edges (>= 5 days): {len(temporal_edges):,}')

Computing temporal co-posting for 263 channels...


Temporal edges:   0%|          | 0/263 [00:00<?, ?it/s]

Temporal edges:  32%|███▏      | 83/263 [00:00<00:00, 776.72it/s]

Temporal edges: 100%|██████████| 263/263 [00:00<00:00, 1787.36it/s]

Temporal co-posting edges (>= 5 days): 19,173


In [6]:
# Combine edges into a single graph
G = nx.Graph()

# Add all channels as nodes
channel_map = channels.set_index('channel_id')['channel_title'].to_dict()
for cid in videos['channel_id'].unique():
    G.add_node(cid, title=channel_map.get(cid, cid))

# Add tag-overlap edges
for (c1, c2), weight in tag_edges.items():
    if G.has_edge(c1, c2):
        G[c1][c2]['weight'] += weight
    else:
        G.add_edge(c1, c2, weight=weight, edge_type='tag')

# Add temporal edges
for (c1, c2), weight in temporal_edges.items():
    norm_weight = weight / 100  # normalize to similar scale as Jaccard
    if G.has_edge(c1, c2):
        G[c1][c2]['weight'] += norm_weight
    else:
        G.add_edge(c1, c2, weight=norm_weight, edge_type='temporal')

print(f'Combined graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Density: {nx.density(G):.6f}')
print(f'Connected components: {nx.number_connected_components(G)}')

Combined graph: 263 nodes, 19173 edges
Density: 0.556497
Connected components: 2


## 3. Network Metrics

In [7]:
# Compute centrality metrics
print('Computing PageRank...')
pagerank = nx.pagerank(G, weight='weight')

print('Computing betweenness centrality...')
betweenness = nx.betweenness_centrality(G, weight='weight', k=min(500, G.number_of_nodes()))

degree_cent = nx.degree_centrality(G)

# Create metrics DataFrame
metrics_df = pd.DataFrame({
    'channel_id': list(G.nodes()),
    'channel_title': [G.nodes[n].get('title', n) for n in G.nodes()],
    'pagerank': [pagerank[n] for n in G.nodes()],
    'betweenness': [betweenness[n] for n in G.nodes()],
    'degree': [degree_cent[n] for n in G.nodes()],
    'num_edges': [G.degree(n) for n in G.nodes()],
})

# Top 20 by PageRank
top20_pr = metrics_df.nlargest(20, 'pagerank')
print('\nTop 20 channels by PageRank:')
top20_pr[['channel_title', 'pagerank', 'betweenness', 'degree', 'num_edges']]

Computing PageRank...


Computing betweenness centrality...



Top 20 channels by PageRank:


,channel_title,pagerank,betweenness,degree,num_edges
182,Alex Christoforou,0.010815,0.000088,0.660305,173
252,HALIDONMUSIC,0.009896,0.000015,0.664122,174
28,super car MO,0.009738,0.000029,0.656489,172
96,Eugenio Monesma - Documentales,0.009560,0.000734,0.698473,183
258,金子吉友の「あつまれニュースの森」,0.009448,0.002034,0.759542,199
64,Anton Petrov,0.009444,0.000175,0.648855,170
111,Pascal Boniface,0.009284,0.000175,0.690840,181
90,Romain Molina,0.008960,0.000409,0.816794,214
11,Hardware Unboxed,0.008949,0.000100,0.782443,205
61,The Rational National,0.008872,0.000608,0.690840,181


In [8]:
# Degree distribution
degrees = [d for _, d in G.degree()]
fig = px.histogram(x=degrees, nbins=50, log_y=True,
    title='Degree Distribution (log scale)',
    labels={'x': 'Degree', 'y': 'Count'})
fig.update_layout(height=400, width=800)
fig.show()

## 4. Community Detection

Using Louvain algorithm for community detection.

In [9]:
# Louvain community detection
partition = community_louvain.best_partition(G, weight='weight', random_state=42)
modularity = community_louvain.modularity(partition, G, weight='weight')

print(f'Modularity: {modularity:.4f}')
print(f'Number of communities: {len(set(partition.values()))}')

# Add cluster info to metrics
metrics_df['cluster_id'] = metrics_df['channel_id'].map(partition)

# Cluster sizes
cluster_sizes = metrics_df['cluster_id'].value_counts().reset_index()
cluster_sizes.columns = ['cluster_id', 'size']
print(f'\nCluster sizes:')
print(cluster_sizes.head(15).to_string())

Modularity: 0.1902
Number of communities: 4

Cluster sizes:
   cluster_id  size
0           0   149
1           1    57
2           2    56
3           3     1


In [10]:
# Top channels per cluster
print('Top 3 channels by PageRank in each of the largest clusters:\n')
top_clusters = cluster_sizes.head(8)['cluster_id'].values
for cid in top_clusters:
    cluster = metrics_df[metrics_df['cluster_id'] == cid].nlargest(3, 'pagerank')
    print(f'--- Cluster {cid} ({cluster_sizes[cluster_sizes["cluster_id"]==cid]["size"].values[0]} channels) ---')
    for _, row in cluster.iterrows():
        print(f'  {row["channel_title"]}: PR={row["pagerank"]:.5f}')
    print()

Top 3 channels by PageRank in each of the largest clusters:

--- Cluster 0 (149 channels) ---
  The Melodramatic Bookworm: PR=0.00794
  葬儀葬式ｃｈ有限会社佐藤葬祭: PR=0.00756
  TFJJ: PR=0.00723

--- Cluster 1 (57 channels) ---
  Alex Christoforou: PR=0.01082
  HALIDONMUSIC: PR=0.00990
  super car MO: PR=0.00974

--- Cluster 2 (56 channels) ---
  Eugenio Monesma - Documentales: PR=0.00956
  金子吉友の「あつまれニュースの森」: PR=0.00945
  Romain Molina: PR=0.00896

--- Cluster 3 (1 channels) ---
  Min0 Taur0: PR=0.00057



## 5. Network Visualization

In [11]:
# Create subgraph of connected nodes for visualization
connected_nodes = [n for n in G.nodes() if G.degree(n) > 0]
G_sub = G.subgraph(connected_nodes)

# If too large, take top nodes by degree
if G_sub.number_of_nodes() > 500:
    top_nodes = sorted(G_sub.nodes(), key=lambda n: G_sub.degree(n), reverse=True)[:500]
    G_sub = G.subgraph(top_nodes)

print(f'Visualization subgraph: {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges')

# Spring layout
pos = nx.spring_layout(G_sub, k=1.5/np.sqrt(G_sub.number_of_nodes()), iterations=50, seed=42)

# Edge traces (sample edges if too many)
edge_x, edge_y = [], []
edges_list = list(G_sub.edges())
if len(edges_list) > 3000:
    edges_sample = np.random.RandomState(42).choice(len(edges_list), 3000, replace=False)
    edges_list = [edges_list[i] for i in edges_sample]

for e in edges_list:
    x0, y0 = pos[e[0]]
    x1, y1 = pos[e[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

# Node traces
node_x = [pos[n][0] for n in G_sub.nodes()]
node_y = [pos[n][1] for n in G_sub.nodes()]
node_colors = [partition.get(n, 0) for n in G_sub.nodes()]
node_sizes = [max(5, pagerank.get(n, 0) * 50000) for n in G_sub.nodes()]
node_text = [
    f"{G.nodes[n].get('title', n)}<br>Cluster: {partition.get(n, '?')}<br>"
    f"PageRank: {pagerank.get(n, 0):.5f}<br>Degree: {G.degree(n)}"
    for n in G_sub.nodes()
]

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.3, color='rgba(150,150,150,0.3)'), hoverinfo='none'))
fig.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers',
    marker=dict(size=node_sizes, color=node_colors, colorscale='Viridis',
                showscale=True, colorbar=dict(title='Cluster'), line=dict(width=0.5, color='white')),
    text=node_text, hoverinfo='text'))

fig.update_layout(title='Channel Network Graph (colored by Louvain cluster, sized by PageRank)',
    showlegend=False, height=800, width=1000,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig.show()

Visualization subgraph: 262 nodes, 19173 edges


In [12]:
# PageRank bar chart
fig = px.bar(top20_pr, x='pagerank', y='channel_title', orientation='h',
    color='pagerank', color_continuous_scale='Blues',
    title='Top 20 Channels by PageRank')
fig.update_layout(height=600, width=900, yaxis=dict(autorange='reversed'))
fig.show()

In [13]:
# Inter-cluster connectivity heatmap
clusters_list = sorted(set(partition.values()))
n_clusters = min(len(clusters_list), 15)  # limit for readability
top_cluster_ids = cluster_sizes.head(n_clusters)['cluster_id'].values

connectivity = np.zeros((n_clusters, n_clusters))
for u, v in G.edges():
    cu, cv = partition.get(u), partition.get(v)
    if cu in top_cluster_ids and cv in top_cluster_ids:
        i = list(top_cluster_ids).index(cu)
        j = list(top_cluster_ids).index(cv)
        connectivity[i][j] += 1
        connectivity[j][i] += 1

fig = px.imshow(connectivity, x=[f'C{c}' for c in top_cluster_ids],
    y=[f'C{c}' for c in top_cluster_ids],
    color_continuous_scale='Blues', title='Inter-Cluster Connectivity Heatmap')
fig.update_layout(height=600, width=700)
fig.show()

## 6. H1 Testing — Coordinated Clusters vs Engagement

**Mann-Whitney U test**: Do channels in the top-3 largest clusters have significantly higher engagement than other channels?

In [14]:
# Get mean engagement per channel
channel_engagement = videos.groupby('channel_id').agg(
    mean_views=('view_count', 'mean'),
    mean_likes=('like_count', 'mean'),
    total_videos=('video_id', 'count'),
).reset_index()

# Merge with cluster info
channel_engagement = channel_engagement.merge(
    metrics_df[['channel_id', 'cluster_id', 'pagerank']], on='channel_id', how='left')

# Top-3 largest clusters
top3_clusters = cluster_sizes.head(3)['cluster_id'].values
in_top3 = channel_engagement[channel_engagement['cluster_id'].isin(top3_clusters)]['mean_views'].dropna()
not_in_top3 = channel_engagement[~channel_engagement['cluster_id'].isin(top3_clusters)]['mean_views'].dropna()

stat, pvalue = stats.mannwhitneyu(in_top3, not_in_top3, alternative='two-sided')

print('=== H1 Test: Coordinated Clusters vs Engagement ===')
print(f'Top-3 cluster channels: {len(in_top3)}')
print(f'Other channels: {len(not_in_top3)}')
print(f'\nMedian views (top-3 clusters): {in_top3.median():,.0f}')
print(f'Median views (other): {not_in_top3.median():,.0f}')
print(f'\nMann-Whitney U statistic: {stat:,.0f}')
print(f'p-value: {pvalue:.4e}')
print(f'\nResult: {"SIGNIFICANT" if pvalue < 0.05 else "NOT SIGNIFICANT"} (alpha=0.05)')
if pvalue < 0.05:
    print('-> H1 SUPPORTED: Coordinated clusters show significantly different engagement.')
else:
    print('-> H1 NOT SUPPORTED at alpha=0.05.')

=== H1 Test: Coordinated Clusters vs Engagement ===
Top-3 cluster channels: 262
Other channels: 1

Median views (top-3 clusters): 2,821
Median views (other): 41

Mann-Whitney U statistic: 256
p-value: 5.3232e-02

Result: NOT SIGNIFICANT (alpha=0.05)
-> H1 NOT SUPPORTED at alpha=0.05.


## 7. Save Outputs

In [15]:
import os
os.makedirs('outputs', exist_ok=True)

# Save cluster assignments
metrics_df.to_csv('outputs/channel_clusters.csv', index=False)
print(f'Saved outputs/channel_clusters.csv ({len(metrics_df):,} rows)')

# Save centrality metrics
metrics_df.to_csv('outputs/channel_centrality_metrics.csv', index=False)
print(f'Saved outputs/channel_centrality_metrics.csv ({len(metrics_df):,} rows)')

print('\nDone!')

Saved outputs/channel_clusters.csv (263 rows)
Saved outputs/channel_centrality_metrics.csv (263 rows)

Done!
